# 01 | Hyperpigmentation — SD2.1 + published cgDDI textual inversions + optional custom LoRA

**Purpose:** Generate five morphology-controlled *full-face clinical-style synthetic images per prompt* using Kaggle T4 GPU.

**Verified resource types:** `hcarrion/hyperpigmentation` and `hcarrion/solar_lentigo` are **textual-inversion adapters**, **not LoRA weight files**. Both target the Stable Diffusion 2.1 base family. They have **not** been validated for faithful full-face melasma/PIH photography.

This notebook (1) loads an SD2.1 base, (2) registers the two optional public textual inversions, (3) optionally loads your custom SD2.1 hyperpigmentation LoRA, (4) encodes long positive and negative prompts through Compel without truncation, and (5) saves five images per prompt and an auditable ZIP.

**Only generated examples; never interpret as actual patient photos or clinically validated lesions.** LoRA weights are **not** included: use Notebook 02 to train them.

**Offline-verified status:** Python cell compilation and CPU-mocked prompt/dataset tests. Actual Kaggle downloads, model loading and CUDA synthesis are not verified locally.

**Audit enhancement:** optional LoRA provenance is checked against the exact base commit before loading.

**Run logs:** every generation attempt saves `notebook_console.log` (including tokenizer or CUDA errors) and `run_log.jsonl`. Successful outputs include per-image SHA-256 manifest, human-review draft CSV and summary JSON. Preserve all artifacts in Kaggle outputs; only privacy-reviewed aggregate statistics should be committed to public GitHub.


Generation outputs are isolated by prompt variant, partition and batch offset. For 1,500 images/class with 5 per authored prompt, run the 210 train prompts and the 45 val and 45 test prompts as three separate partitions (1,050/225/225 images), keeping held-out prompt families out of training. Different offsets can safely produce separate batch folders without silently overwriting previous runs.


## 1 | Install fixed package versions

Use a new Kaggle notebook, **GPU T4** and **Internet ON**. The installer leaves Kaggle's CUDA-enabled PyTorch unchanged. If an earlier session loaded conflicting modules, restart once and rerun from the top.

In [ ]:
import subprocess, sys
REQUIREMENTS = [
    'diffusers==0.35.1', 'transformers==4.56.2', 'accelerate==1.10.1',
    'peft==0.17.1', 'huggingface_hub==0.34.4',
    'safetensors==0.6.2', 'ftfy==6.3.1', 'compel==2.3.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
print('Dependency setup finished. Use a fresh Kaggle kernel if imported modules remain stale.')

In [ ]:
import json, csv, re, time, gc, zipfile, hashlib
from pathlib import Path
from IPython.display import display
from PIL import Image
import torch, diffusers, transformers, accelerate, compel as compel_package
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from compel import Compel, ReturnedEmbeddingsType, DiffusersTextualInversionManager
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle Settings.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'diffusers:', diffusers.__version__,
      'transformers:', transformers.__version__, 'compel:', getattr(compel_package,'__version__','installed'))

## 2 | Configuration — edit here

`ADAPTER_MODE` is set independently **for each prompt** in Section 3. `none` = no published disease token; `broad` = published hyperpigmentation textual inversion; `solar` = solar-lentigo textual inversion. This is a deliberately *per-prompt choice*: a broad pigment model is **not** a validated melasma/PIH model, and solar lentigo is not PIH.

`LORA_SOURCE`: leave blank for the public-adapter pilot. After training, point to the SD2.1 LoRA `.safetensors` file or its directory. Keep the **same SD2.1 base family** for LoRA training and inference.

In [ ]:
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'  # accessible community SD2.1 512 base mirror; verify provenance/license
PUBLIC_ADAPTERS = {
    'broad': 'hcarrion/hyperpigmentation',
    'solar': 'hcarrion/solar_lentigo',
}
LORA_SOURCE = ''  # e.g. '/kaggle/input/your-hyperpigmentation-lora/pytorch_lora_weights.safetensors'
LORA_SCALE = 0.8  # experimental, not clinically optimized
OUT_DIR = Path('/kaggle/working/hyperpigmentation_generated')
IMAGES_PER_PROMPT = 5
WIDTH = HEIGHT = 512
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 52031
MAX_CONTENT_TOKENS = 150
MAX_CLIP_CHUNKS = 2
NEGATIVE_PROMPT = ('cartoon, illustration, artificial makeup, skin painting, heavy airbrushing, '
                   'wax skin, disfigured facial anatomy, extra face, blurry, watermark, text')
assert 1 <= IMAGES_PER_PROMPT <= 100
assert 0 <= LORA_SCALE <= 2
assert WIDTH % 8 == 0 and HEIGHT % 8 == 0
OUT_DIR.mkdir(parents=True,exist_ok=True)

## 3. Load protected authored morphology prompts
Reads `prompts/` JSON files (300 prompts: 210 train, 45 val, 45 test; 60 independent families). Defaults to 5 train prompts; `PROMPT_LIMIT=None` selects all training prompts. Preserves original negative prompts and emits CSV+JSON review records. All generated images remain unapproved pending human QA.


In [ ]:
# Use project-authored 300 morphology prompts and protected prompt-family split.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'
PROMPT_VARIANT='morphology'  # use 'generic' for E1, morphology for E2
PROMPT_OFFSET=0
PROMPT_LIMIT=5   # Kaggle pilot; None uses all 210 train prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Hyperpigmentation',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET,variant=PROMPT_VARIANT)
# Keep E1 generic and E2 morphology images in separate run directories; never overwrite.
OUT_DIR=(Path(OUT_DIR)/('prompt_'+PROMPT_VARIANT)/PROMPT_PARTITION/
         ('offset_'+str(PROMPT_OFFSET)+'_count_'+str(len(PROMPTS))))
if OUT_DIR.exists():
    raise FileExistsError(f'Generation output already exists: {OUT_DIR}; start a fresh run directory')
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))


In [ ]:
# Capture generation console, warnings and failed diffusion execution to disk.
from dermsynreal.kaggle_runs import start_notebook_logging
GENERATION_CONSOLE=start_notebook_logging(OUT_DIR,kind='diffusion_generation',
    config={'prompt_partition':PROMPT_PARTITION,'prompt_variant':PROMPT_VARIANT,
            'prompt_count':len(PROMPTS),'class_name':'Hyperpigmentation',
            'images_per_prompt':IMAGES_PER_PROMPT},repo_root=DERMSYNREAL_REPO)
print('Generation console file:',GENERATION_CONSOLE)


## 4 | Load base model, discover actual textual-inversion tokens, attach optional LoRA

**No guessed special token strings.** The cell discovers the special token registered by each published adapter. `DiffusersTextualInversionManager` is passed to Compel for multi-vector embeddings. If an adapter cannot be loaded, stop with its actual error rather than silently generating with a different model.

In [ ]:
base_revision = model_info(BASE_MODEL).sha
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, revision=base_revision, torch_dtype=torch.float16, use_safetensors=True
)

adapter_tokens, adapter_revisions = {}, {}
needed_modes = {x.get('adapter_mode','none') for x in PROMPTS} - {'none'}
for mode in ('broad','solar'):
    if mode not in needed_modes:
        continue
    model_id = PUBLIC_ADAPTERS[mode]
    revision = model_info(model_id).sha
    previous_vocab = set(pipe.tokenizer.get_vocab())
    pipe.load_textual_inversion(model_id, revision=revision)
    new_tokens = sorted(set(pipe.tokenizer.get_vocab()) - previous_vocab)
    if not new_tokens:
        raise RuntimeError(f'{model_id}: textual inversion loaded but no new tokenizer token was detected')
    base_token_candidates = [t for t in new_tokens if not re.search(r'_\d+$', t)]
    token = min(base_token_candidates or new_tokens, key=len)
    if token in adapter_tokens.values():
        raise RuntimeError('Two disease adapters registered the same token: cannot disambiguate')
    adapter_tokens[mode] = token
    adapter_revisions[mode] = revision
    print('Loaded',mode,model_id,'token:',repr(token),'additional tokens:',len(new_tokens))

if str(LORA_SOURCE).strip():
    source = str(LORA_SOURCE).strip()
    source_path = Path(source)
    # When training exports provenance, refuse to silently combine weights
    # with a different SD2.1 base model or revision.
    provenance_dir = source_path.parent if source_path.is_file() else source_path
    provenance_file = provenance_dir / 'training_provenance.json'
    if provenance_file.is_file():
        training_provenance = json.loads(provenance_file.read_text(encoding='utf-8'))
        if training_provenance.get('base_model') != BASE_MODEL:
            raise RuntimeError('LoRA base model mismatch; use the original training backbone')
        if training_provenance.get('base_revision') != base_revision:
            raise RuntimeError('LoRA base revision mismatch; use the same weight snapshot')
        print('Verified matching LoRA training backbone and exact revision.')
    else:
        print('WARNING: No LoRA training_provenance.json found. Verify its SD2.1 base manually.')
    if source_path.is_file():
        if source_path.suffix.lower() != '.safetensors':
            raise ValueError('Local LoRA file must use .safetensors extension')
        pipe.load_lora_weights(str(source_path.parent), weight_name=source_path.name, adapter_name='facial_hp')
    else:
        pipe.load_lora_weights(source, adapter_name='facial_hp')
    pipe.set_adapters(['facial_hp'], adapter_weights=[LORA_SCALE])
    print('Enabled custom hyperpigmentation LoRA:',source)
else:
    print('Public-adapter baseline; custom LoRA not loaded.')

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.to('cuda')
pipe.text_encoder.eval()
print('Base revision:',base_revision)

## 5 | Compel — 150 content tokens / two SD2.1 CLIP chunks

For SD2.1, a CLIP window has 77 positions **including BOS/EOS**, leaving **75 content tokens**. We reject inputs over **150 content tokens** after applying textual-inversion expansion. This includes negative prompts. Compel encodes both prompts without truncation and pads resulting embeddings to equal length.

**Training captions** in Notebook 02 follow the official SD2.1 single-window rule (75 content tokens); 150-token Compel encoding is specifically an **inference** feature.

In [ ]:
assert pipe.tokenizer.model_max_length == 77, 'Unexpected tokenizer context size; check SD2.1 base.'
assert MAX_CONTENT_TOKENS == 150 and MAX_CLIP_CHUNKS == 2
CONTENT_PER_CHUNK = pipe.tokenizer.model_max_length - 2
assert MAX_CONTENT_TOKENS <= CONTENT_PER_CHUNK * MAX_CLIP_CHUNKS

# Required when using published textual inversions with Compel (especially multi-vector embeddings).
ti_manager = DiffusersTextualInversionManager(pipe)
compel_encoder = Compel(
    tokenizer=pipe.tokenizer,
    text_encoder=pipe.text_encoder,
    textual_inversion_manager=ti_manager,
    returned_embeddings_type=ReturnedEmbeddingsType.LAST_HIDDEN_STATES_NORMALIZED,
    truncate_long_prompts=False,
    requires_pooled=False,
    device='cuda',
)

def build_prompt(entry):
    mode = entry.get('adapter_mode','none')
    if mode not in {'none',*PUBLIC_ADAPTERS}:
        raise ValueError(f'Invalid adapter mode: {mode}')
    prompt = entry['text'].strip()
    if mode != 'none':
        special = adapter_tokens.get(mode)
        if not special:
            raise RuntimeError(f'Adapter {mode} was not loaded')
        if special not in prompt:
            prompt = prompt + ', ' + special
    return prompt

def content_tokens(prompt):
    # Diffusers expands multi-vector textual-inversion placeholders into all registered token IDs.
    expanded = pipe.maybe_convert_prompt(prompt, pipe.tokenizer)
    ids = pipe.tokenizer(expanded, truncation=False, add_special_tokens=False)['input_ids']
    return len(ids)

def enforce_token_limit(prompt, label):
    if not isinstance(prompt,str) or not prompt.strip():
        raise ValueError(f'{label} is empty')
    n = content_tokens(prompt)
    if n > MAX_CONTENT_TOKENS:
        raise ValueError(f'{label}: {n} content tokens; limit is {MAX_CONTENT_TOKENS}. Shorten it.')
    print(f'{label}: {n} content tokens, {max(1,(n + CONTENT_PER_CHUNK - 1)//CONTENT_PER_CHUNK)} chunk(s)')
    return n

# Before generating anything, validate all actual expanded prompts.
validated_prompts = {}
for item in PROMPTS:
    text_prompt = build_prompt(item)
    enforce_token_limit(text_prompt,item['id'])
    validated_prompts[item['id']] = text_prompt
for entry in PROMPTS:
    enforce_token_limit(entry.get('negative_prompt',NEGATIVE_PROMPT),entry['id']+'/negative')

## 6 | Generate five images per prompt; show previews; log model/adapter revisions

The dataset ZIP contains PNG images and `metadata.jsonl`. All subtype labels are **requested prompts**, not expert-validated diagnoses. Filter duplicates and clinically implausible images before using the results for classification.

In [ ]:
def safe_id(text):
    return re.sub(r'[^a-zA-Z0-9_-]+', '_', text).strip('_')[:72] or 'prompt'

all_metadata = []
start = time.time()
for idx, entry in enumerate(PROMPTS):
    positive = validated_prompts[entry['id']]
    negative = entry.get('negative_prompt',NEGATIVE_PROMPT)
    with torch.no_grad():
        positive_embeds = compel_encoder(positive)
        negative_embeds = compel_encoder(negative)
        positive_embeds, negative_embeds = compel_encoder.pad_conditioning_tensors_to_same_length(
            [positive_embeds, negative_embeds]
        )
    if positive_embeds.shape != negative_embeds.shape:
        raise RuntimeError('Positive and negative conditioning embeddings have mismatched shape')
    if positive_embeds.ndim != 3 or positive_embeds.shape[1] > pipe.tokenizer.model_max_length*MAX_CLIP_CHUNKS:
        raise RuntimeError(f'Unexpected Compel embedding shape: {tuple(positive_embeds.shape)}')
    positive_embeds = positive_embeds.to(device='cuda', dtype=pipe.unet.dtype)
    negative_embeds = negative_embeds.to(device='cuda', dtype=pipe.unet.dtype)
    folder = OUT_DIR / safe_id(entry['id'])
    folder.mkdir(parents=True,exist_ok=True)
    gallery = []
    for j in range(IMAGES_PER_PROMPT):
        seed = BASE_SEED + idx*1000 + j
        rng = torch.Generator(device='cuda').manual_seed(seed)
        with torch.inference_mode():
            image = pipe(
                prompt_embeds=positive_embeds,
                negative_prompt_embeds=negative_embeds,
                num_inference_steps=STEPS,
                guidance_scale=GUIDANCE,
                height=HEIGHT, width=WIDTH,
                generator=rng,
            ).images[0]
        filepath = folder / f'{safe_id(entry["id"])}_seed{seed}.png'
        image.save(filepath)
        gallery.append(image)
        mode = entry.get('adapter_mode','none')
        all_metadata.append({
            'file': filepath.relative_to(OUT_DIR).as_posix(),
            'prompt_id': entry['id'],
            'requested_subtype': entry.get('subtype','unspecified'),
            'requested_adapter_mode': mode,
            'actual_special_token': adapter_tokens.get(mode),
            'prompt': positive,
            'negative_prompt': negative,
            'positive_content_tokens': content_tokens(positive),
            'negative_content_tokens': content_tokens(negative),
            'embedding_positions': positive_embeds.shape[1],
            'seed': seed,
            'base_model': BASE_MODEL,
            'base_revision': base_revision,
            'textual_inversion_model': PUBLIC_ADAPTERS.get(mode),
            'textual_inversion_revision': adapter_revisions.get(mode),
            'lora_source': LORA_SOURCE or None,
            'lora_scale': LORA_SCALE if LORA_SOURCE else None,
            'resolution': [WIDTH, HEIGHT], 'steps': STEPS, 'guidance': GUIDANCE,
            'morphology_review': 'pending', 'duplicate_review': 'pending',
        })
    contact = Image.new('RGB', (WIDTH*len(gallery),HEIGHT))
    for k, picture in enumerate(gallery):
        contact.paste(picture,(k*WIDTH,0))
    show_width = min(contact.width,1200)
    display(contact.resize((show_width,int(contact.height*show_width/contact.width))))
    del positive_embeds, negative_embeds
    gc.collect()

with (OUT_DIR / 'metadata.jsonl').open('w',encoding='utf-8') as out:
    for row in all_metadata:
        out.write(json.dumps(row,ensure_ascii=False)+'\n')
print('Generated',len(all_metadata),'images in',round(time.time()-start,1),'seconds')

In [ ]:
# Export reproducible per-image CSV, SHA hashes, review draft and run provenance.
summary=generation_manifest(all_metadata,OUT_DIR/'reports',root=OUT_DIR,class_name='Hyperpigmentation',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL,prompt_type=PROMPT_VARIANT)
append_run_log(OUT_DIR,kind='generation',config={'class':'Hyperpigmentation','prompt_partition':PROMPT_PARTITION,'prompt_type':PROMPT_VARIANT,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('CSV:',OUT_DIR/'reports/generation_manifest.csv','Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv')

## 7 | Export a Kaggle-downloadable ZIP

In [ ]:
ZIP_PATH = Path('/kaggle/working') / ('hyperpigmentation_'+PROMPT_VARIANT+'_'+PROMPT_PARTITION+'_offset'+str(PROMPT_OFFSET)+'_n'+str(len(PROMPTS))+'_k'+str(IMAGES_PER_PROMPT)+'.zip')
if ZIP_PATH.exists(): raise FileExistsError('Generation ZIP for this partition and offset already exists; choose a new run')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as bundle:
    for fp in sorted(OUT_DIR.rglob('*')):
        if fp.is_file():
            bundle.write(fp,fp.relative_to(OUT_DIR).as_posix())
assert len(all_metadata)==len(PROMPTS)*IMAGES_PER_PROMPT
print('ZIP ready:',ZIP_PATH,'MiB:',round(ZIP_PATH.stat().st_size / 2**20,2))

## References and limits
- [cgDDI hyperpigmentation textual inversion](https://huggingface.co/hcarrion/hyperpigmentation)
- [cgDDI solar lentigo textual inversion](https://huggingface.co/hcarrion/solar_lentigo)
- [SD2.1 community base mirror](https://huggingface.co/sd2-community/stable-diffusion-2-1-base)
- [Compel v2.3.1](https://pypi.org/project/compel/2.3.1/)
- [Diffusers LoRA guide](https://huggingface.co/docs/diffusers/v0.35.0/training/lora)

Neither public adapter is clinically validated for full-face photographs. The custom LoRA and its compatibility must be tested with actual Kaggle GPU inference before any generation-quality claims.